# Providers 00 - Auto (CLI)

Notebook CLI paralelo a `tutorials/providers/00_auto.ipynb`.

**Objetivo:** Resolver automáticamente el primer Provider listo.

Este notebook no llama factories de Agentic Systems directamente: ejecuta el
entrypoint CLI real, conserva la salida Rich y valida después el JSON del mismo
contrato.


## Cómo se ejecuta

La forma portable es `python -m agentic_systems.cli ...`. Después de instalar
el wheel, el entrypoint equivalente es `agentic-systems ...`.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def _repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio.")


ROOT = _repo_root()
CLI = [sys.executable, "-m", "agentic_systems.cli"]


def run_cli(*args: str, expected: int = 0) -> str:
    env = os.environ.copy()
    source_path = str(ROOT / "src")
    env["PYTHONPATH"] = (
        source_path
        if not env.get("PYTHONPATH")
        else source_path + os.pathsep + env["PYTHONPATH"]
    )
    command = [*CLI, *args]
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )
    print("$ " + " ".join(command))
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    assert completed.returncode == expected, completed.stderr
    return completed.stdout


def run_cli_json(*args: str) -> dict:
    return json.loads(run_cli(*args, "--json"))


def assert_rich(output: str, title: str) -> None:
    assert title in output
    ascii_box = "+" in output and "|" in output
    unicode_box = "─" in output and "│" in output
    assert ascii_box or unicode_box


## 1) Salida humana Rich

La celda conserva stdout y comprueba título y bordes. Esto detecta tablas o
paneles truncados, además del exit code.


In [2]:
rich_output = run_cli(*['doctor'])
assert_rich(rich_output, 'Agentic Systems Doctor')


$ C:\Python314\python.exe -m agentic_systems.cli doctor
+-------------------------- Agentic Systems Doctor ---------------------------+
| Agentic Systems 2.1.0                                                       |
| Python: 3.14.2                                                              |
| Engines: bedrock-runtime, openai-runtime, ollama-runtime, python-runtime,   |
| vllm-runtime                                                                |
| .env loaded: True                                                           |
+-----------------------------------------------------------------------------+
       Environment                         Optional Dependencies               
 VLLM_BASE_URL    missing                boto3           available             
 OPENAI_API_KEY   set                    langgraph       available             
 OLLAMA_BASE_URL  set                    openai          available             
 OLLAMA_MODEL     set                    openai-agents   availab

## 2) Contrato de máquina

La misma ruta se ejecuta con `--json` para afirmar campos y cardinalidad sin
parsear la presentación Rich.


In [3]:
payload = run_cli_json(*['runtime', '--provider', 'auto', '--allow-python-fallback'])
assert payload["selected_provider"] in {"python-runtime", "openai-runtime", "ollama-runtime", "vllm-runtime", "bedrock-runtime"}
payload


$ C:\Python314\python.exe -m agentic_systems.cli runtime --provider auto --allow-python-fallback --json
{
  "api_key_configured": false,
  "configuration": {
    "bedrock": {
      "aws_profile_configured": false,
      "aws_region": "us-east-2",
      "bedrock_api_key_configured": true,
      "credentials_configured": true
    }
  },
  "endpoint": null,
  "fallback_provider": "openai-runtime",
  "mode": "auto",
  "model": "us.amazon.nova-pro-v1:0",
  "preferred_provider": "bedrock-runtime",
  "provider_priority": [
    "bedrock-runtime",
    "openai-runtime",
    "vllm-runtime",
    "ollama-runtime",
    "python-runtime"
  ],
  "reason": "AWS authentication and region detected",
  "region": "us-east-2",
  "scheduler": {
    "backoff_s": 0.0,
    "max_concurrency": 1,
    "max_retries": 0,
    "max_tool_calls": 5,
    "max_turns": 6,
    "timeout_s": 60.0
  },
  "selected_provider": "bedrock-runtime"
}


{'api_key_configured': False,
 'configuration': {'bedrock': {'aws_profile_configured': False,
   'aws_region': 'us-east-2',
   'bedrock_api_key_configured': True,
   'credentials_configured': True}},
 'endpoint': None,
 'fallback_provider': 'openai-runtime',
 'mode': 'auto',
 'model': 'us.amazon.nova-pro-v1:0',
 'preferred_provider': 'bedrock-runtime',
 'provider_priority': ['bedrock-runtime',
  'openai-runtime',
  'vllm-runtime',
  'ollama-runtime',
  'python-runtime'],
 'reason': 'AWS authentication and region detected',
 'region': 'us-east-2',
 'scheduler': {'backoff_s': 0.0,
  'max_concurrency': 1,
  'max_retries': 0,
  'max_tool_calls': 5,
  'max_turns': 6,
  'timeout_s': 60.0},
 'selected_provider': 'bedrock-runtime'}

## Resultado e interpretación

Rich responde a lectura humana; JSON responde a automatización. Ambos nacen del
mismo comando y del mismo escenario público. Un estado `not-run` conserva el
motivo, pero no cuenta como evidencia live.
